# 02 - PDF Ingestion and Document Chunking

## Objective

In this notebook, we will:

1. Load a real PDF
2. Extract text from the PDF
3. Inspect the extracted text
4. Understand why chunking is necessary
5. Implement a basic chunking strategy
6. Experiment with different chunk sizes
7. Prepare chunks for embedding and retrieval

from pathlib import Path
from pypdf import PdfReader

In [1]:
from pathlib import Path
from pypdf import PdfReader

In [2]:
PDF_PATH = Path("../data/Lora.pdf")

print("PDF exists:", PDF_PATH.exists())
print("PDF path:", PDF_PATH)

PDF exists: True
PDF path: ..\data\Lora.pdf


5. Extract text from the PDF

In [3]:
reader = PdfReader(PDF_PATH)

print("Number of pages:", len(reader.pages))

Number of pages: 26


In [4]:
page_texts = []

for page_number, page in enumerate(reader.pages):
    text = page.extract_text()

    page_texts.append({
        "page": page_number + 1,
        "text": text or ""
    })

print("Pages extracted:", len(page_texts))

Pages extracted: 26


In [5]:
print(page_texts[0]["text"][:3000])


LORA: L OW-R ANK ADAPTATION OF LARGE LAN-
GUAGE MODELS
Edward Hu∗ Yelong Shen∗ Phillip Wallis Zeyuan Allen-Zhu
Yuanzhi Li Shean Wang Lu Wang Weizhu Chen
Microsoft Corporation
{edwardhu, yeshe, phwallis, zeyuana,
yuanzhil, swang, luw, wzchen }@microsoft.com
yuanzhil@andrew.cmu.edu
(Version 2)
ABSTRACT
An important paradigm of natural language processing consists of large-scale pre-
training on general domain data and adaptation to particular tasks or domains. As
we pre-train larger models, full ﬁne-tuning, which retrains all model parameters,
becomes less feasible. Using GPT-3 175B as an example – deploying indepen-
dent instances of ﬁne-tuned models, each with 175B parameters, is prohibitively
expensive. We propose Low-Rank Adaptation, or LoRA, which freezes the pre-
trained model weights and injects trainable rank decomposition matrices into each
layer of the Transformer architecture, greatly reducing the number of trainable pa-
rameters for downstream tasks. Compared to GPT-3 175B ﬁn

6. Why can't we embed the entire PDF?

This is an extremely important RAG concept.

Suppose your paper contains:

20,000 words

You could theoretically create:

PDF
 ↓
20,000 words
 ↓
ONE embedding

But that's a bad retrieval strategy.

Imagine the user asks:

"What dataset did the authors use?"

We don't want to retrieve an embedding representing the entire paper.

We want something like:

Chunk 1 → Introduction
Chunk 2 → Related Work
Chunk 3 → Dataset
Chunk 4 → Methodology
Chunk 5 → Experiments
...

Then the query can retrieve the specific relevant chunk.

So:

               WHOLE DOCUMENT

                    ↓

       ┌────────────┬────────────┐
       ↓            ↓            ↓
    Chunk 1      Chunk 2      Chunk 3
       ↓            ↓            ↓
   Embedding    Embedding    Embedding

This is why chunking is fundamental to RAG.

First simple chunking implementation

In [6]:
def create_chunks(text, chunk_size=500, overlap=50):
    chunks = []

    start = 0

    while start < len(text):
        end = start + chunk_size

        chunk = text[start:end]

        if chunk.strip():
            chunks.append(chunk.strip())

        start += chunk_size - overlap

    return chunks

In [7]:
full_text = "\n".join(
    page["text"]
    for page in page_texts
)

In [8]:
chunks = create_chunks(
    full_text,
    chunk_size=1000,
    overlap=100
)

print("Number of chunks:", len(chunks))

Number of chunks: 92


8. Inspect the chunks

In [9]:
for i, chunk in enumerate(chunks[:5]):
    print("=" * 80)
    print(f"CHUNK {i}")
    print("=" * 80)
    print(chunk[:1000])

CHUNK 0
LORA: L OW-R ANK ADAPTATION OF LARGE LAN-
GUAGE MODELS
Edward Hu∗ Yelong Shen∗ Phillip Wallis Zeyuan Allen-Zhu
Yuanzhi Li Shean Wang Lu Wang Weizhu Chen
Microsoft Corporation
{edwardhu, yeshe, phwallis, zeyuana,
yuanzhil, swang, luw, wzchen }@microsoft.com
yuanzhil@andrew.cmu.edu
(Version 2)
ABSTRACT
An important paradigm of natural language processing consists of large-scale pre-
training on general domain data and adaptation to particular tasks or domains. As
we pre-train larger models, full ﬁne-tuning, which retrains all model parameters,
becomes less feasible. Using GPT-3 175B as an example – deploying indepen-
dent instances of ﬁne-tuned models, each with 175B parameters, is prohibitively
expensive. We propose Low-Rank Adaptation, or LoRA, which freezes the pre-
trained model weights and injects trainable rank decomposition matrices into each
layer of the Transformer architecture, greatly reducing the number of trainable pa-
rameters for downstream tasks. Compared to GPT-3

9. Understand chunk_size

We used:

chunk_size=1000

But there's an important detail:

Our current implementation uses characters, not tokens.

So:

chunk_size = 1000

means approximately:

1000 characters

not:

1000 tokens

That's intentional for our first experiment.

Later we'll implement token-aware and semantic chunking.

10. Understand overlap

We used:

overlap=100

Suppose:

Chunk 1:
AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA
                  ↓
             overlap
                  ↓
Chunk 2:
                  AAAAAAAAAAAAAAAA

The end of one chunk overlaps with the beginning of the next.

Why?

Because important information can sit across a boundary.

For example:

Chunk 1:
"The proposed model consists of three components:
encoder, decoder, and..."

Chunk 2:
"...cross-attention module. The encoder..."

Without overlap, context can get split.

With overlap:

Chunk 1
       └──────────────┐
                      ↓
                 Chunk 2

some surrounding context is preserved.

In [10]:
# Chunk 1:
# AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA
#                   ↓
#              overlap
#                   ↓
# Chunk 2:
#                   AAAAAAAAAAAAAAAA

11. Let's experiment with chunk sizes

This is important.

Don't just assume:

chunk_size = 500

is correct.

In [11]:
chunk_sizes = [300, 500, 1000, 1500, 2000]

for size in chunk_sizes:
    temp_chunks = create_chunks(
        full_text,
        chunk_size=size,
        overlap=int(size * 0.1)
    )

    print(
        f"Chunk size: {size:4d} | "
        f"Chunks: {len(temp_chunks)}"
    )

Chunk size:  300 | Chunks: 306
Chunk size:  500 | Chunks: 184
Chunk size: 1000 | Chunks: 92
Chunk size: 1500 | Chunks: 62
Chunk size: 2000 | Chunks: 46


12. Why chunk size matters

There's a tradeoff.

Very small chunks
PDF
 ↓
tiny pieces

Advantages:

precise retrieval
less irrelevant context

Problems:

context can be lost
chunks may not contain enough information
Very large chunks
PDF
 ↓
huge pieces

Advantages:

more context

Problems:

retrieval becomes less precise
more irrelevant text
more tokens sent to the LLM

So we're looking for a balance:

Too small ←────── Good chunk ──────→ Too large

Later we'll measure this rather than guessing.

13. Create metadata

This is an important improvement.

Don't store chunks as just strings.

Eventually we want:

{
    "text": "...",
    "page": 5,
    "chunk_id": 23,
    "source": "sample.pdf"
}

This becomes extremely useful for citations.

Let's create a better chunk representation.

First, modify our approach to process each page separately:

In [12]:
def create_page_chunks(page_texts, chunk_size=1000, overlap=100):
    chunks = []

    for page in page_texts:
        text = page["text"]

        start = 0

        while start < len(text):
            end = start + chunk_size

            chunk_text = text[start:end].strip()

            if chunk_text:
                chunks.append({
                    "text": chunk_text,
                    "page": page["page"]
                })

            start += chunk_size - overlap

    return chunks

In [13]:
chunks = create_page_chunks(
    page_texts,
    chunk_size=1000,
    overlap=100
)

print("Total chunks:", len(chunks))

Total chunks: 104


14. Inspect metadata

In [14]:
for i, chunk in enumerate(chunks[:5]):
    print("=" * 80)
    print("Chunk ID:", i)
    print("Page:", chunk["page"])
    print("Text:")
    print(chunk["text"][:500])

Chunk ID: 0
Page: 1
Text:
LORA: L OW-R ANK ADAPTATION OF LARGE LAN-
GUAGE MODELS
Edward Hu∗ Yelong Shen∗ Phillip Wallis Zeyuan Allen-Zhu
Yuanzhi Li Shean Wang Lu Wang Weizhu Chen
Microsoft Corporation
{edwardhu, yeshe, phwallis, zeyuana,
yuanzhil, swang, luw, wzchen }@microsoft.com
yuanzhil@andrew.cmu.edu
(Version 2)
ABSTRACT
An important paradigm of natural language processing consists of large-scale pre-
training on general domain data and adaptation to particular tasks or domains. As
we pre-train larger models, full ﬁ
Chunk ID: 1
Page: 1
Text:
reatly reducing the number of trainable pa-
rameters for downstream tasks. Compared to GPT-3 175B ﬁne-tuned with Adam,
LoRA can reduce the number of trainable parameters by 10,000 times and the
GPU memory requirement by 3 times. LoRA performs on-par or better than ﬁne-
tuning in model quality on RoBERTa, DeBERTa, GPT-2, and GPT-3, despite hav-
ing fewer trainable parameters, a higher training throughput, and, unlike adapters,
no additional inf

15. Add a chunk ID

In [15]:
for chunk_id, chunk in enumerate(chunks):
    chunk["chunk_id"] = chunk_id
    chunk["source"] = PDF_PATH.name

In [16]:
chunks[0]

{'text': 'LORA: L OW-R ANK ADAPTATION OF LARGE LAN-\nGUAGE MODELS\nEdward Hu∗ Yelong Shen∗ Phillip Wallis Zeyuan Allen-Zhu\nYuanzhi Li Shean Wang Lu Wang Weizhu Chen\nMicrosoft Corporation\n{edwardhu, yeshe, phwallis, zeyuana,\nyuanzhil, swang, luw, wzchen }@microsoft.com\nyuanzhil@andrew.cmu.edu\n(Version 2)\nABSTRACT\nAn important paradigm of natural language processing consists of large-scale pre-\ntraining on general domain data and adaptation to particular tasks or domains. As\nwe pre-train larger models, full ﬁne-tuning, which retrains all model parameters,\nbecomes less feasible. Using GPT-3 175B as an example – deploying indepen-\ndent instances of ﬁne-tuned models, each with 175B parameters, is prohibitively\nexpensive. We propose Low-Rank Adaptation, or LoRA, which freezes the pre-\ntrained model weights and injects trainable rank decomposition matrices into each\nlayer of the Transformer architecture, greatly reducing the number of trainable pa-\nrameters for downstream task

16. Let's test it 

In [17]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

In [18]:
from src.loader import load_pdf
from src.chunker import create_page_chunks

pages = load_pdf("../data/Lora.pdf")

print("Pages:", len(pages))

Pages: 26


In [19]:
chunks = create_page_chunks(
    pages,
    chunk_size=1000,
    overlap=100
)

print("Chunks:", len(chunks))

Chunks: 104


In [20]:
chunks[0]

{'page': 1,
 'text': 'LORA: L OW-R ANK ADAPTATION OF LARGE LAN-\nGUAGE MODELS\nEdward Hu∗ Yelong Shen∗ Phillip Wallis Zeyuan Allen-Zhu\nYuanzhi Li Shean Wang Lu Wang Weizhu Chen\nMicrosoft Corporation\n{edwardhu, yeshe, phwallis, zeyuana,\nyuanzhil, swang, luw, wzchen }@microsoft.com\nyuanzhil@andrew.cmu.edu\n(Version 2)\nABSTRACT\nAn important paradigm of natural language processing consists of large-scale pre-\ntraining on general domain data and adaptation to particular tasks or domains. As\nwe pre-train larger models, full ﬁne-tuning, which retrains all model parameters,\nbecomes less feasible. Using GPT-3 175B as an example – deploying indepen-\ndent instances of ﬁne-tuned models, each with 175B parameters, is prohibitively\nexpensive. We propose Low-Rank Adaptation, or LoRA, which freezes the pre-\ntrained model weights and injects trainable rank decomposition matrices into each\nlayer of the Transformer architecture, greatly reducing the number of trainable pa-\nrameters for dow